# cluetask

> Clue tasks for the Phase 11 board: a hidden integer fixed by modular clues ("x mod p = r") over distinct small primes, chosen so that every clue is needed and missing one leaves at least two candidates; the feasible (clues, range) pairs.

In [ ]:
#| default_exp cluetask

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math
from dataclasses import dataclass
from itertools import combinations

import numpy as np


PRIMES = (2, 3, 5, 7, 11, 13)
CLUE_PAIRS = ((2, 30), (2, 60), (3, 105), (3, 210), (4, 210))      # the feasible (m, R) pairs with PRIMES


def primeSets(m, R):
    "Sets of m primes with product >= R and every (m - 1)-subset product <= R / 2 (each clue needed)."
    return [c for c in combinations(PRIMES, m)
            if math.prod(c) >= R and all(math.prod(s) <= R / 2 for s in combinations(c, m - 1))]


@dataclass
class ClueTask:
    primes: tuple      # m distinct primes
    x: int             # the hidden answer, 0 <= x < R
    R: int

    def clue(self, j):
        "Clue j as (p, r): 'x mod p = r'."
        return self.primes[j], self.x % self.primes[j]

    def check(self, a):
        return isinstance(a, (int, np.integer)) and not isinstance(a, bool) and 0 <= a < self.R and int(a) == self.x

    def candidates(self, known):
        "The y in [0, R) consistent with the known residues {j: r} (which may include false ones)."
        return [y for y in range(self.R) if all(y % self.primes[j] == r for j, r in known.items())]


def makeClueTask(rng, m, R):
    "A clue task: a feasible prime set for (m, R) drawn uniformly, and x uniform in [0, R)."
    sets = primeSets(m, R)
    if not sets:
        raise ValueError(f'no prime set fits m = {m}, R = {R}')
    return ClueTask(tuple(sets[int(rng.integers(len(sets)))]), int(rng.integers(R)), int(R))

## Tests

In [ ]:
import math
import numpy as np
from explRL.cluetask import *

# AC-2: every feasible prime set gives a unique answer; any m - 1 clues leave at least two candidates
for m, R in CLUE_PAIRS:
    sets = primeSets(m, R)
    assert sets, (m, R)
    for ps in sets:
        for x in (0, R // 3, R - 1):
            t = ClueTask(ps, x, R)
            allc = {j: t.clue(j)[1] for j in range(m)}
            assert t.candidates(allc) == [x] and t.check(x) and not t.check(x + 1 if x + 1 < R else 0)
            for drop in range(m):
                part = {j: r for j, r in allc.items() if j != drop}
                assert len(t.candidates(part)) >= 2, (ps, x, drop)
assert [len(primeSets(m, R)) for m, R in CLUE_PAIRS] == [8, 4, 1, 4, 1] and not primeSets(2, 210)
a, b = makeClueTask(np.random.default_rng(3), 3, 210), makeClueTask(np.random.default_rng(3), 3, 210)
assert a == b and a.primes in primeSets(3, 210) and 0 <= a.x < 210
assert not ClueTask((3, 5, 7), 23, 105).check(True) and not ClueTask((3, 5, 7), 23, 105).check(23.0)
assert ClueTask((3, 5, 7), 23, 105).clue(2) == (7, 2)          # the worked example: x mod 7 = 2

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()